# Notebook 1 — LOBSTER Data Exploration & LOB Dynamics

**Project:** Adaptive Execution via Online Parameter Estimation  
**Data:** AAPL · NASDAQ · 2012-06-21 · Level-5 LOBSTER  
**Author:** Changkui Wu (FSU Financial Mathematics PhD, 2026)

---

## Overview

This notebook loads the raw LOBSTER tick data, constructs the key state variables
used in the Obizhaeva–Wang (2013) execution model, and visualises the intraday
behaviour of the limit order book.

### What is the OW model?

Obizhaeva & Wang (2013) model the limit order book (LOB) as a block-shaped queue
whose *deviation* from steady state decays exponentially:

$$dD_t = -r \, D_t \, dt - k \, dX_t + \sigma \, dB_t$$

| Symbol | Meaning |
|--------|---------|
| $D_t = A_t - (F_t + s/2)$ | Ask deviation above steady-state level |
| $r > 0$ | **Resilience** — speed of LOB recovery |
| $k = 1/q - \lambda$ | Price impact coefficient |
| $\sigma$ | Diffusion noise of the LOB |
| $dX_t$ | Trade flow (our control) |

The key insight of OW: the *optimal execution strategy depends only on $r$*, not
on the static spread or depth.  Since $r$ is unobservable, we need to estimate it
online — which is exactly what the PhD thesis addresses.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter

from lobster import load_lobster, resample, estimate_params

# ── style ──────────────────────────────────────────────────────────────────
DARK, GRID = '#0f1117', '#1a1d2e'
TEXT, BLUE  = '#e0e0e0', '#4a9eff'
GREEN, ORANGE, RED = '#50fa7b', '#ffb86c', '#ff5555'
PURPLE, CYAN = '#bd93f9', '#8be9fd'

plt.rcParams.update({
    'figure.facecolor': DARK, 'axes.facecolor': GRID,
    'axes.edgecolor': '#333355', 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': '#252540',
    'grid.linewidth': 0.6, 'legend.facecolor': GRID,
    'legend.labelcolor': TEXT, 'font.size': 10,
})

DATA_DIR = os.path.join('..', 'data')
MSG_FILE = os.path.join(DATA_DIR, 'AAPL_2012-06-21_34200000_57600000_message_5.csv')
OBK_FILE = os.path.join(DATA_DIR, 'AAPL_2012-06-21_34200000_57600000_orderbook_5.csv')

print("Loading LOBSTER data...")
lob = load_lobster(MSG_FILE, OBK_FILE, ema_tau=30.0)
print(f"  Events:       {len(lob.time):,}")
print(f"  Time range:   {lob.time[0]/3600:.2f}h – {lob.time[-1]/3600:.2f}h")
print(f"  Price range:  ${lob.mid.min():.2f} – ${lob.mid.max():.2f}")
print(f"  Executions:   {len(lob.exec_time):,}")
print(f"  Mean spread:  {lob.spread.mean()*100:.2f} cents")


## 1. Price and Fundamental Value

The **mid-quote** $V_t = (A_t + B_t)/2$ is the raw market price signal.
The **fundamental value** $F_t$ is proxied by an exponential moving average (EMA)
of the mid-quote with timescale $\tau = 30$ seconds.  The LOB deviation
$D_t = A_t - (F_t + s/2)$ measures how far the ask price has been pushed
above its steady-state level by recent trades.


In [ ]:
t_h   = lob.time / 3600.0
step  = max(1, len(lob.time) // 3000)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('AAPL LOB Dynamics — 2012-06-21', fontsize=13, color=TEXT)

# ── Panel 1: price ────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(t_h[::step], lob.mid[::step],   color=BLUE,   lw=0.8, label='Mid-quote $V_t$')
ax.plot(t_h[::step], lob.Ft[::step],    color=ORANGE, lw=1.0, alpha=0.8, label='EMA $F_t$ (τ=30s)')
# large trade markers
exec_h = lob.exec_time / 3600.0
large  = lob.exec_size >= np.percentile(lob.exec_size, 95)
for et in exec_h[large][::5]:
    ax.axvline(et, color=RED, alpha=0.1, lw=0.5)
ax.set_ylabel('Price ($)')
ax.legend(loc='upper right', fontsize=9)
ax.set_title('Mid-quote and Fundamental Value Proxy', color=TEXT, fontsize=10)

# ── Panel 2: LOB deviation ────────────────────────────────────────────────
ax = axes[1]
ax.plot(t_h[::step], lob.Dt[::step], color=CYAN, lw=0.5, alpha=0.85)
ax.axhline(0, color=TEXT, lw=0.8, ls='--', alpha=0.4)
ax.fill_between(t_h[::step], lob.Dt[::step], 0,
                where=lob.Dt[::step]>0, color=RED,    alpha=0.15)
ax.fill_between(t_h[::step], lob.Dt[::step], 0,
                where=lob.Dt[::step]<0, color=GREEN,  alpha=0.15)
ax.set_ylabel('$D_t$ ($)')
ax.set_title('LOB Deviation  $D_t = A_t - (F_t + s/2)$', color=TEXT, fontsize=10)

# ── Panel 3: spread & depth ───────────────────────────────────────────────
ax  = axes[2]
axb = ax.twinx()
ax.plot(t_h[::step], lob.spread[::step]*100, color=GREEN,  lw=0.6, alpha=0.8, label='Spread (¢)')
axb.plot(t_h[::step], lob.ask_size[::step],   color=PURPLE, lw=0.5, alpha=0.6, label='Depth (shares)')
ax.set_ylabel('Spread (¢)',       color=GREEN)
axb.set_ylabel('L1 Depth (sh)',   color=PURPLE)
axb.tick_params(colors=PURPLE)
ax.set_xlabel('Time of day (hours)')
ax.set_title('Bid-Ask Spread & Level-1 Ask Depth', color=TEXT, fontsize=10)
lines = ax.get_lines() + axb.get_lines()
ax.legend(lines, [l.get_label() for l in lines], fontsize=9, loc='upper right')

for a in axes:
    a.set_xlim(t_h[0], t_h[-1])

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb1_lob_dynamics.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()
print("Saved → outputs/nb1_lob_dynamics.png")


## 2. Trade Size Distribution

LOBSTER event types 4 and 5 are visible and hidden limit-order executions,
i.e. actual market trades.  The top-5% by size are "large trades" that push
$D_t$ noticeably and trigger the resilience recovery we want to estimate.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Trade Statistics', color=TEXT, fontsize=12)

# Distribution
ax = axes[0]
sizes = lob.exec_size
thr95 = np.percentile(sizes, 95)
ax.hist(sizes[sizes < thr95], bins=60, color=BLUE, alpha=0.75,
        density=True, edgecolor='none', label='< 95th pct')
ax.hist(sizes[sizes >= thr95], bins=20, color=RED,  alpha=0.75,
        density=True, edgecolor='none', label='≥ 95th pct (large)')
ax.axvline(thr95, color=RED, lw=1.5, ls='--')
ax.set_xlabel('Trade size (shares)')
ax.set_ylabel('Density')
ax.set_title('Trade Size Distribution')
ax.legend(fontsize=9)

# Intraday trade intensity
ax = axes[1]
bins  = np.arange(9.5, 16.1, 0.25)
cnts, edges = np.histogram(lob.exec_time/3600, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2
ax.bar(centers, cnts, width=0.22, color=BLUE, alpha=0.75, edgecolor='none')
ax.set_xlabel('Time of day (hours)')
ax.set_ylabel('Executions per 15-min bin')
ax.set_title('Intraday Trade Intensity (U-shape)')
ax.axvline(9.5,  color=RED, lw=1, ls='--', alpha=0.5)
ax.axvline(16.0, color=RED, lw=1, ls='--', alpha=0.5)

for a in axes:
    a.set_facecolor(GRID)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb1_trade_stats.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()

print(f"Total executions:         {len(sizes):,}")
print(f"Large trade threshold:    {thr95:.0f} shares  (95th percentile)")
print(f"Large trades:             {(sizes>=thr95).sum():,}")
print(f"Mean trade size:          {sizes.mean():.1f} shares")
print(f"Median trade size:        {np.median(sizes):.1f} shares")


## 3. Post-Trade LOB Recovery

The core empirical test of the OW model: after a large buy trade, the ask
price $A_t$ is pushed up (large $D_t$), then decays back toward steady state
as new liquidity providers refill the book.

OW models this decay as $D_t \approx D_0 e^{-rt}$.  The slope on a
log-scale plot against time gives us $-r$.


In [ ]:
large_idx  = np.where(lob.exec_size >= thr95)[0]
# Map back to full timeline indices
all_exec   = np.where(np.isin(lob.time,
                 lob.exec_time[lob.exec_size >= thr95]))[0]

window = 45.0
curves = []
for li in large_idx[:300]:
    t0 = lob.exec_time[li]
    D0 = np.interp(t0, lob.time, lob.Dt)
    if abs(D0) < 0.01:
        continue
    mask = (lob.time > t0) & (lob.time <= t0 + window)
    if mask.sum() < 8:
        continue
    curves.append((lob.time[mask] - t0,
                   lob.Dt[mask] / D0))

t_th = np.linspace(0, window, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Post-Large-Trade LOB Recovery', color=TEXT, fontsize=12)

for ax, logy in zip(axes, [False, True]):
    for tc, Dc in curves[:30]:
        ax.plot(tc, Dc, color=ORANGE, alpha=0.20, lw=0.8)
    # Overlay theoretical decay for a range of r values
    for r_plot, c in [(0.02, CYAN), (0.036, RED), (0.1, GREEN)]:
        y = np.exp(-r_plot * t_th)
        ax.plot(t_th, y, lw=1.8, ls='--', color=c, label=f'r={r_plot}')
    ax.axhline(0, color=TEXT, lw=0.5, ls='--', alpha=0.4)
    ax.set_xlabel('Seconds after large trade')
    ax.set_ylabel('$D_t / D_0$' + (' (log scale)' if logy else ''))
    ax.set_title('Linear scale' if not logy else 'Log scale  (slope = -r)')
    ax.set_xlim(0, window)
    if logy:
        ax.set_yscale('log')
        ax.set_ylim(1e-2, 10)
    else:
        ax.set_ylim(-1.5, 2.0)
    ax.legend(fontsize=9)
    ax.set_facecolor(GRID)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb1_recovery.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()
print(f"Recovery curves plotted: {len(curves)}")


## 4. Summary Statistics

Summary of key LOB statistics for the trading day.
These will be used to calibrate the OW simulation in Notebook 3.


In [ ]:
params = estimate_params(lob, dt=1.0)

print("=" * 50)
print("LOB Summary Statistics — AAPL 2012-06-21")
print("=" * 50)
print(f"  Mean mid-quote:    ${lob.mid.mean():.2f}")
print(f"  Mean spread:       {lob.spread.mean()*100:.3f} cents")
print(f"  Median L1 depth:   {np.median(lob.ask_size):.0f} shares")
print(f"  Total executions:  {len(lob.exec_time):,}")
print(f"  Large trades (95%): {(lob.exec_size >= thr95).sum():,}")
print()
print("OW Model Parameters (MLE estimates):")
print(f"  r     = {params['r']:.4f} /s   (half-life {np.log(2)/params['r']:.1f}s)")
print(f"  sigma = {params['sigma']:.5f} $/sqrt(s)")
print(f"  q     = {params['q']:.0f} shares")
print(f"  lam   = {params['lam']:.5f} $/share^2")
print(f"  k     = {params['k']:.5f} $/share^2")
print("=" * 50)
print()
print("→ These parameters are passed to Notebook 2 (estimation) and")
print("  Notebook 3 (simulation) as the calibrated ground truth.")
